# Feature Engineering
Extract sliding-window features from the combined dataset for ML classification.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# input/output files and sliding window settings
INPUT_CSV  = "ultrasonic_combined_clean.csv"
OUTPUT_CSV = "ultrasonic_features.csv"

WINDOW_SIZE = 10   # samples (~2 seconds at 200ms sampling)
STEP_SIZE   = 5    # sliding overlap

df = pd.read_csv(INPUT_CSV)

In [3]:
features = []

# process each activity class separately
for label in df["label"].unique():
    df_label = df[df["label"] == label].reset_index(drop=True)

    for i in range(0, len(df_label) - WINDOW_SIZE, STEP_SIZE):
        window = df_label.iloc[i:i + WINDOW_SIZE]

        distances = window["distance_cm"].values

        # skip windows with too many invalid readings
        if np.mean(distances < 0) > 0.5:
            continue

        mean_dist  = np.mean(distances)
        std_dist   = np.std(distances)
        min_dist   = np.min(distances)
        max_dist   = np.max(distances)
        range_dist = max_dist - min_dist

        rate_of_change    = np.mean(np.abs(np.diff(distances)))
        presence_duration = WINDOW_SIZE * 0.2  # seconds

        features.append([
            mean_dist,
            std_dist,
            range_dist,
            rate_of_change,
            presence_duration,
            label
        ])

In [4]:
feature_df = pd.DataFrame(
    features,
    columns=[
        "mean_distance",
        "std_distance",
        "range_distance",
        "rate_of_change",
        "presence_duration",
        "label"
    ]
)

feature_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved features to {OUTPUT_CSV}")
print("Label distribution:")
print(feature_df["label"].value_counts())

Saved features to ultrasonic_features.csv
Label distribution:
label
sedentary    117
active       117
Name: count, dtype: int64
